# L13b: Synthetic Markets of Many Stocks
Previously, we built a jump hidden Markov model of the daily growth rate of one asset. How do we simulate many stocks that move together, as real stocks in the same market and sector do?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Test the single-asset model out of sample:__ Compare paths simulated from the jump hidden Markov model of `SPY` with the daily growth rates that came after its fitting period. Turn the jump episodes off and compare both models, including their volatility clustering, on the same data.
> * __Compose a synthetic market of many stocks:__ Combine each stock's baseline with market, sector, and stock-specific parts to build synthetic stocks that move together like real ones. Describe the single-asset jump hidden Markov models that supply the market, sector, and stock-specific draws.
> * __Size the stock-specific draw and sample the universe:__ Choose each stock's residual scale so that its sector and stock-specific draws keep the variance of its single index residual. Draw one synthetic path of every stock, holding each stock's market sensitivity fixed or letting it drift.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * An out-of-sample test compares a fitted model with data from after its fitting period. We test the jump hidden Markov model of `SPY`, with and without jump episodes, on the growth rates of 2025 and 2026.
> * We model a synthetic stock's growth rate as $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + \psi_{i}\,u_{i}$, where $\alpha_{i}$ is the stock's baseline. The shared market and sector parts make stocks move together. The stock-specific draw $u_{i}$ averages zero and is the growth that neither the market nor the sector explains.
> * The residual scale $\psi_{i}$ matches each stock's variance to the data. A drifting $\beta_{i}$ lets the stock's market sensitivity change over time.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The tests that compare the `SPY` model with the out-of-sample data.
> * How we compute the sector loadings and the residual scales, and the drifting $\beta$ option.
> * The sampling algorithm of the sector factor model, step by step.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!

___


## Examples

__We will work through these examples in lecture:__

> [▶ Test the `SPY` model out of sample](CHEME-5660-L13b-Example-JumpHMM-SPY-OutOfSample-Fall-2026.ipynb). In this example, we load the jump hidden Markov model of `SPY` fitted to daily growth rates from 2014 to 2024. We compare its simulated paths with the growth rates of 2025 and 2026, which the model has not seen. We then turn the jump episodes off, score both models on the same data, and compare the volatility clustering of simulated and real growth rates.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we use the sector factor model to build a universe of `SPY` and 375 stocks from all 11 sectors. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, turn off the sector part, and save the fitted model.

___


## Review: Synthetic Markets with Jump Hidden Markov Models
Previously, we built synthetic markets from jump hidden Markov models. The model bins the daily growth rate into states of equal probability and estimates the transition matrix by counting moves between states.

Each simulated day, the chain takes one step. The model then draws the day's growth rate from the Student-t distribution of the new state:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

Jump episodes start at random and last several days. On each episode day, the state comes from the most negative or the most positive states. These runs of large moves add the volatility clustering that the chain alone misses.

Let's test the jump hidden Markov model of `SPY` on data it has not seen:

> __Example:__
>
> [▶ Test the `SPY` model out of sample](CHEME-5660-L13b-Example-JumpHMM-SPY-OutOfSample-Fall-2026.ipynb). In this example, we load the jump hidden Markov model of `SPY` fitted to daily growth rates from 2014 to 2024. We compare its simulated paths with the growth rates of 2025 and 2026, which the model has not seen. We then turn the jump episodes off, score both models on the same data, and compare the volatility clustering of simulated and real growth rates.

__Today:__ We extend the jump hidden Markov model from one asset to many stocks that move together.

___


## Company Profile: AQR Capital Management
[AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us) is a quantitative investment management firm. It was founded in New York City in 1998 and moved its headquarters to Greenwich, Connecticut, in 2004. The firm describes [its approach](https://www.aqr.com/What-We-Do/Our-Approach) as "systematic investing grounded in economic theory."

> __What does the firm do?__
>
> * __Betting against beta:__ In [Betting Against Beta](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta), published in 2014, AQR's Andrea Frazzini and Lasse H. Pedersen found that stocks with high beta tend to have low alpha. A portfolio that bought low-beta stocks with borrowed money and sold high-beta stocks short earned positive risk-adjusted returns.

__Explore further:__

* __Jobs and internships:__ AQR's [internships page](https://www.aqr.com/Our-Firm/Greenwich-Internships) describes a 10-week summer program in Greenwich. In November 2026, the [careers site](https://careers.aqr.com/) listed 2027 summer analyst roles in research, engineering, and business development.
* __Video:__ Watch [About AQR](https://www.youtube.com/embed/946GJrhyZtM) and [Internships at AQR](https://www.youtube.com/embed/Oow2UktGKcY) on the firm's YouTube channel.

__Connection to today's lecture:__ The betting-against-beta portfolio ranks stocks by their market sensitivity $\beta_{i}$, which sets the market part of our sector factor model. With a synthetic universe, we could test a betting-against-beta portfolio on many plausible market paths.

___


## Many Assets: The Sector Factor Model
We extend the single-asset jump hidden Markov model to many stocks with a factor model. We start from the single index model and the covariance matrix it generates. Then we add a sector factor that supplies the covariance the single index model misses.

The resulting sector factor model splits each stock's daily growth rate into a fixed baseline plus market, sector, and stock-specific parts. We build the many-stock model from single-asset jump hidden Markov models: one for the market, one for each sector, and one for each stock.

Previously, the multiple asset GBM model mixed independent Gaussian shocks through a loading matrix. Here, the independent inputs are paths of these single-asset models. Each stock's loadings set how much of the market, sector, and stock-specific paths enter its growth rate.

### The Single Index Model and Its Covariance
Previously, we used the single index model to build the covariance matrix of the growth rates of many stocks. The single index model splits the growth rate of stock $i$ on day $t$ into a baseline, a market part, and a single index residual:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\varepsilon_{i}(t)}_{\text{single index residual}}
$$

Here $g_{m}(t)$ is the growth rate of `SPY`, and the single index residual $\varepsilon_{i}(t)$ is the growth that the market does not explain. The model assumes that the residuals have zero mean, are uncorrelated with the market, and are uncorrelated across stocks. On the diagonal of the covariance matrix, the variance of stock $i$ is given by:

$$
\begin{align*}
\operatorname{Var}(g_{i}) &= \operatorname{Var}\left(\alpha_{i}+\beta_{i}\,g_{m}+\varepsilon_{i}\right) && \text{substitute the single index model}\\
&= \beta_{i}^{2}\operatorname{Var}(g_{m})+2\,\beta_{i}\,\underbrace{\operatorname{Cov}(g_{m},\varepsilon_{i})}_{=0}+\operatorname{Var}(\varepsilon_{i}) && \text{expand, second assumption}\\
&= \beta_{i}^{2}\operatorname{Var}(g_{m})+\operatorname{Var}(\varepsilon_{i})
\end{align*}
$$

Off the diagonal, for two different stocks $i\neq j$, taking the covariance of their single index models gives:

$$
\begin{align*}
\operatorname{Cov}(g_{i},g_{j}) &= \operatorname{Cov}\left(\alpha_{i}+\beta_{i}\,g_{m}+\varepsilon_{i},\;\alpha_{j}+\beta_{j}\,g_{m}+\varepsilon_{j}\right) && \text{substitute both single index models}\\
&= \beta_{i}\beta_{j}\operatorname{Var}(g_{m})+\beta_{i}\,\underbrace{\operatorname{Cov}(g_{m},\varepsilon_{j})}_{=0}+\beta_{j}\,\underbrace{\operatorname{Cov}(\varepsilon_{i},g_{m})}_{=0}+\underbrace{\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j})}_{=0} && \text{expand, second and third assumptions}\\
&= \beta_{i}\beta_{j}\operatorname{Var}(g_{m})
\end{align*}
$$

Collecting the market sensitivities of the $n$ stocks in the vector $\boldsymbol{\beta}$ and placing these entries in a matrix gives:

$$
\begin{align*}
\mathbf{\Sigma}_{g}
&=\begin{bmatrix}
\beta_{1}^{2}\operatorname{Var}(g_{m})+\operatorname{Var}(\varepsilon_{1}) & \beta_{1}\beta_{2}\operatorname{Var}(g_{m}) & \cdots\\
\beta_{2}\beta_{1}\operatorname{Var}(g_{m}) & \beta_{2}^{2}\operatorname{Var}(g_{m})+\operatorname{Var}(\varepsilon_{2}) & \cdots\\
\vdots & \vdots & \ddots
\end{bmatrix} && \text{place the entries}\\
&=\operatorname{Var}(g_{m})\begin{bmatrix}
\beta_{1}^{2} & \beta_{1}\beta_{2} & \cdots\\
\beta_{2}\beta_{1} & \beta_{2}^{2} & \cdots\\
\vdots & \vdots & \ddots
\end{bmatrix}
+\begin{bmatrix}
\operatorname{Var}(\varepsilon_{1}) & 0 & \cdots\\
0 & \operatorname{Var}(\varepsilon_{2}) & \cdots\\
\vdots & \vdots & \ddots
\end{bmatrix} && \text{split each entry}\\
&=\underbrace{\operatorname{Var}(g_{m})\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top}}_{\text{market}}+\underbrace{\mathbf{\Sigma}_{\varepsilon}}_{\text{residual}} && \text{name the two matrices}
\end{align*}
$$

The market matrix $\boldsymbol{\beta}\boldsymbol{\beta}^{\top}$ has the entry $\beta_{i}\beta_{j}$ in row $i$ and column $j$, so every entry has a market part. The residual matrix $\mathbf{\Sigma}_{\varepsilon}$ is the covariance matrix of the single index residuals. In the single index model, the residual matrix is diagonal, because the residuals of two different stocks are uncorrelated.

__Why not use the single index model?__ The first published version of this method used the single index model to simulate 424 stocks ([Alswaidan and Varner, 2026](https://arxiv.org/abs/2603.10202)). Every simulated stock shared one market draw from the model of `SPY`, and each stock's residual was drawn independently each day. On the fitting data, the simulated stocks passed the Kolmogorov–Smirnov test much less often than the simulated `SPY` did.

The paper traced this gap to the single index model and its residual draws, which did not fully capture the tails of each stock's growth rates. Here, we simulate the stock-specific part of each growth rate with its own jump hidden Markov model. The single index model also misses how stocks in the same sector move together.

In the data, the single index residuals of two stocks in the same sector, such as two banks, usually move together, with $\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j})>0$. The independent residual draws set this covariance to zero, as the third assumption does. Simulated stocks in the same sector then move together less than real stocks do.

### Adding a Sector Factor
Previously, we studied __factor models__, which describe the growth rate in terms of factors. For stock $i$ on day $t$, a general factor model is given by:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\sum_{j}\delta_{ij}\,f_{j}(t)}_{\text{other factors}} + \underbrace{\varepsilon_{i}(t)}_{\text{residual}}
$$

Here $\alpha_{i}$ is the stock-specific baseline (intercept), $g_{m}(t)$ is the market growth rate, and $f_{j}(t)$ are other factors, such as interest rates or inflation. The coefficients $\beta_{i}$ and $\delta_{ij}$ measure sensitivity to these factors. The __residual__ $\varepsilon_{i}(t)$ is the growth left after subtracting the baseline and factor contributions.

> __Definition (sector factor model):__ The single index model keeps only the market factor. A sector factor model adds one factor for each sector of the Global Industry Classification Standard (GICS), such as Financials or Energy. Each sector factor describes how the single index residuals of the stocks in its sector move together.
> 
> Let's model the daily growth rate of stock $i$ in sector $s$ as the sum of four parts:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \overbrace{\underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}} + \underbrace{u_{i}(t)}_{\text{stock-specific}}}^{\text{single index residual }\varepsilon_{i}(t)}
> $$
>
> Here $g_{m}(t)$ is the growth rate of `SPY`, $f_{s}(t)$ is the growth rate of sector $s$ with the market part removed, and $u_{i}(t)$ is the growth that neither the market nor the sector explains.

The sector factor model splits the single index residual of stock $i$ into a sector part and a stock-specific part, $\varepsilon_{i}(t)=\delta_{i}\,f_{s}(t)+u_{i}(t)$. In the simulation, we draw the market, each sector factor, and each stock-specific residual independently. The market part of the covariance matrix therefore stays the same, and only the residual matrix $\mathbf{\Sigma}_{\varepsilon}$ changes.

The independent draws make the sector factors of different sectors uncorrelated. They also make each stock-specific residual uncorrelated with every sector factor and with the other stock-specific residuals. For stock $i$ in sector $s$ and a different stock $j$ in sector $r$, taking the covariance of their sector splits gives:

$$
\begin{align*}
\operatorname{Cov}(\varepsilon_{i},\varepsilon_{j}) &= \operatorname{Cov}\left(\delta_{i}\,f_{s}+u_{i},\;\delta_{j}\,f_{r}+u_{j}\right) && \text{substitute both sector splits}\\
&= \delta_{i}\delta_{j}\operatorname{Cov}(f_{s},f_{r})+\delta_{i}\,\underbrace{\operatorname{Cov}(f_{s},u_{j})}_{=0}+\delta_{j}\,\underbrace{\operatorname{Cov}(u_{i},f_{r})}_{=0}+\underbrace{\operatorname{Cov}(u_{i},u_{j})}_{=0} && \text{expand, independent draws}\\
&= \begin{cases}\delta_{i}\delta_{j}\operatorname{Var}(f_{s}) & \text{same sector, } r=s\\ 0 & \text{different sectors, } r\neq s\end{cases} && \text{independent sector draws}
\end{align*}
$$

On the diagonal, we size the stock-specific draws below so that each entry keeps the variance $\operatorname{Var}(\varepsilon_{i})$ of the single index residual, as in the single index model. Placing these entries in the residual matrix gives one block for each sector.

> __Example:__ Suppose stocks 1 and 2 are in sector $s$, and stocks 3 and 4 are in sector $r$. The residual matrix of the four stocks is given by:
>
> $$
> \mathbf{\Sigma}_{\varepsilon}=
> \left[\begin{array}{cc|cc}
> {\color{#1f6fb4} \operatorname{Var}(\varepsilon_{1})} & {\color{#1f6fb4} \delta_{1}\delta_{2}\operatorname{Var}(f_{s})} & {\color{#999999} 0} & {\color{#999999} 0}\\
> {\color{#1f6fb4} \delta_{1}\delta_{2}\operatorname{Var}(f_{s})} & {\color{#1f6fb4} \operatorname{Var}(\varepsilon_{2})} & {\color{#999999} 0} & {\color{#999999} 0}\\ \hline
> {\color{#999999} 0} & {\color{#999999} 0} & {\color{#d4720f} \operatorname{Var}(\varepsilon_{3})} & {\color{#d4720f} \delta_{3}\delta_{4}\operatorname{Var}(f_{r})}\\
> {\color{#999999} 0} & {\color{#999999} 0} & {\color{#d4720f} \delta_{3}\delta_{4}\operatorname{Var}(f_{r})} & {\color{#d4720f} \operatorname{Var}(\varepsilon_{4})}
> \end{array}\right]
> \begin{array}{l}\left.\vphantom{\begin{matrix}\operatorname{Var}(\varepsilon_{1})\\\operatorname{Var}(\varepsilon_{1})\end{matrix}}\right\}{\color{#1f6fb4} \text{ sector } s}\\ \left.\vphantom{\begin{matrix}\operatorname{Var}(\varepsilon_{1})\\\operatorname{Var}(\varepsilon_{1})\end{matrix}}\right\}{\color{#d4720f} \text{ sector } r}\end{array}
> $$
>
> The single index model has the same diagonal and zeros everywhere else.

### The Stock-Specific Part
The stock-specific draw is a path of daily growth rates simulated from the single-asset jump hidden Markov model of stock $i$. We multiply the draw by the residual scale $\psi_{i}$ and add $\psi_{i}\,u_{i}(t)$ to the growth rate of stock $i$. The sector draw and the stock-specific draw are independent.

We choose the residual scale so that the variance of the simulated single index residual equals $\operatorname{Var}(\varepsilon_{i})$, the diagonal entry of the residual matrix:

$$
\boxed{\psi_{i}^{2} = \frac{\operatorname{Var}(\varepsilon_{i})-\delta_{i}^{2}\operatorname{Var}(f_{s})}{\operatorname{Var}(u_{i})}}
$$

The numerator is the variance that the sector part leaves to the stock-specific draw.

### Comparing the Two Covariance Matrices
In general, each sector adds one block. With the residual scales chosen, the covariance matrices of the growth rates of the two models are:

$$
\begin{aligned}
\text{single index model:}\quad\mathbf{\Sigma}_{g} &= \underbrace{\operatorname{Var}(g_{m})\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top}}_{\text{market}} + \underbrace{\operatorname{diag}\left(\operatorname{Var}(\varepsilon_{1}),\dotsc,\operatorname{Var}(\varepsilon_{n})\right)}_{\text{residual}}\\[6pt]
\text{sector factor model:}\quad\mathbf{\Sigma}_{g} &= \underbrace{\operatorname{Var}(g_{m})\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top}}_{\text{market}} + \underbrace{\sum_{s}\operatorname{Var}(f_{s})\,\boldsymbol{\delta}_{s}\boldsymbol{\delta}_{s}^{\top}}_{\text{sector}} + \underbrace{\operatorname{diag}\left(\psi_{1}^{2}\operatorname{Var}(u_{1}),\dotsc,\psi_{n}^{2}\operatorname{Var}(u_{n})\right)}_{\text{stock-specific}}
\end{aligned}
$$

Here the vector $\boldsymbol{\delta}_{s}$ holds the sector loading $\delta_{i}$ of each stock in sector $s$ and zeros for the other stocks. Both matrices have the same market term. They also have the same diagonal, because the residual scale makes $\delta_{i}^{2}\operatorname{Var}(f_{s})+\psi_{i}^{2}\operatorname{Var}(u_{i})=\operatorname{Var}(\varepsilon_{i})$.

The sector factor model adds the sector blocks. Inside each block, the off-diagonal entries $\delta_{i}\delta_{j}\operatorname{Var}(f_{s})$ are the residual covariances that the single index model sets to zero. Pairs of stocks in different sectors keep the single index covariance $\beta_{i}\beta_{j}\operatorname{Var}(g_{m})$.

### Computing the Parts from the Data
We compute the parts from the data in the order below. The [derivation notebook](advanced/sector_factor/CHEME-5660-L13b-Derivation-SectorFactor-Parts-Fall-2026.ipynb) shows how we get each result.

* __Sector factor:__ The sector factor is the average single index residual of the $n_{s}$ stocks in sector $s$. Substituting the single index model of each stock into this average gives:

  $$
  \begin{align*}
  f_{s}(t) &= \frac{1}{n_{s}}\sum_{j\in s}\varepsilon_{j}(t) && \text{average single index residual}\\
  &= \frac{1}{n_{s}}\sum_{j\in s}\left(g_{j}(t)-\alpha_{j}-\beta_{j}\,g_{m}(t)\right) && \text{single index fit of each stock } j\\
  &= \frac{1}{n_{s}}\sum_{j\in s}g_{j}(t)-\frac{1}{n_{s}}\sum_{j\in s}\alpha_{j}-\left(\frac{1}{n_{s}}\sum_{j\in s}\beta_{j}\right)g_{m}(t) && \text{split the sum, and factor out } g_{m}(t)\\
  &= \underbrace{\bar{g}_{s}(t)}_{\text{sector average}}-\underbrace{\bar{\alpha}_{s}}_{\text{baseline}}-\underbrace{\bar{\beta}_{s}\,g_{m}(t)}_{\text{market}} && \text{name the three averages}
  \end{align*}
  $$

  Each bar is an average over the $n_{s}$ stocks in the sector: $\bar{g}_{s}(t)$ of their growth rates on day $t$, $\bar{\alpha}_{s}$ of their baselines, and $\bar{\beta}_{s}$ of their market sensitivities.

  The sector factor is the sector's average growth rate with the baseline and the market part removed. A market move therefore enters the growth rate only once, through $\beta_{i}\,g_{m}(t)$.

* __Sector loading:__ We estimate $\hat{\delta}_{i}$ by least squares, as the slope of $\varepsilon_{i}(t)$ on the sector factor computed without stock $i$. We then multiply the slopes of each sector by one number $c_{s}$, so the sector loading is $\delta_{i}=c_{s}\,\hat{\delta}_{i}$. We choose $c_{s}$ so that the off-diagonal entries of the sector block match the residual covariances in the data on average.
* __Stock-specific residual:__ Removing the sector part leaves the stock-specific residual $u_{i}(t)=\varepsilon_{i}(t)-\delta_{i}\,f_{s}(t)$, which averages zero in the data. We then compute the residual scale $\psi_{i}$ from the variances of $\varepsilon_{i}(t)$, $f_{s}(t)$, and $u_{i}(t)$.

The sector factors and the stock-specific residuals are daily growth rates, like the growth rates of `SPY`. We fit a single-asset jump hidden Markov model to each of these growth rate series, the same way we fit and tuned the model of `SPY`.

### Sampling Algorithm
The algorithm below draws one synthetic path of the whole universe over $T$ days.

__Initialize__: Given the market model (the jump hidden Markov model of `SPY` that we tested out of sample), the fitted sector models, and the fitted stock-specific models; each stock's baseline $\alpha_{i}$, market sensitivity $\beta_{i}$, sector loading $\delta_{i}$, and residual scale $\psi_{i}$; each stock's real starting price $S_{i}(0)$; the time step $\Delta t=1/252$ year (one trading day); and the number of days $T$.

1. __Shared draws:__ Draw one market path $g_{m}(1),\dotsc,g_{m}(T)$, shared by all stocks, from the market model. For each sector $s$ __do__: draw one factor path $f_{s}(1),\dotsc,f_{s}(T)$, shared by the stocks in that sector, from its sector model. Use independent random draws to generate the different paths.
2. __Stock-specific draws:__ For each stock $i$ __do__: draw a path $u_{i}(1),\dotsc,u_{i}(T)$ from the stock-specific model of stock $i$.
3. __Compose:__ For each stock $i$ in sector $s$ and each day $t = 1,2,\dots,T$ __do__: compose the growth rate $g_{i}(t) \gets \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,u_{i}(t)$, and compound the price, starting from $S_{i}(0)$: $S_{i}(t) \gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The market path $g_{m}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t = 1,\dotsc,T$.

__Note:__ With the drifting β option below, $\beta_{i}$ becomes $\beta_{i}(t)$ in step 3. We also reduce $\psi_{i}$ to keep each stock's variance unchanged.

### An option: a drifting β
Holding $\beta_{i}$ fixed assumes that stock $i$'s sensitivity to the market growth rate stays the same over a synthetic path. Instead, we can let the sensitivity change over time around its single index estimate $\hat{\beta}_{i}$. Each day, we update it with:

$$
\beta_{i}(t) = \hat{\beta}_{i}
+ \underbrace{(1-a)\left(\beta_{i}(t-1)-\hat{\beta}_{i}\right)}_{\text{retained deviation}}
+ \underbrace{a\,k_{i}\,z_{i}(t)}_{\text{new shock}}.
$$

Here $0<a<1$ is the exponential moving average weight. A small $a$ keeps most of the previous day's deviation from $\hat{\beta}_{i}$. The shocks $z_{i}(t)$ are independent standard normal draws, and $k_{i}\geq 0$ sets their size.

In the example, we choose $k_{i}$ from how much daily β estimates vary in the real data. These estimates also contain sampling noise. The market part is now $\beta_{i}(t)\,g_{m}(t)$.

Let's look at an example:

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we use the sector factor model to build a universe of `SPY` and 375 stocks from all 11 sectors. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, turn off the sector part, and save the fitted model.

___

## Summary
We tested the jump hidden Markov model of `SPY` on data it had not seen. We then extended the model from one asset to a universe of stocks that move together, as real stocks in the same market and sector do.

> __Key Takeaways:__
>
> * __Testing the single-asset model out of sample:__ We compared paths simulated from the jump hidden Markov model of `SPY` with the growth rates of 2025 and 2026, which came after its fitting period. The paths with a jump episode failed the distribution tests more often, and their volatility clustering lasted longer than in the data.
> * __Many stocks from single-asset models:__ We built the synthetic universe from single-asset jump hidden Markov models: one for the market, one for each sector, and one for each stock. Every stock shares the market draw, and every stock in a sector shares the sector draw. The shared sector draw adds the residual covariance of stocks in the same sector, which the single index model sets to zero.
> * __Residual scale and a drifting market sensitivity:__ We sized the stock-specific draw so that, together with the sector draw, it keeps the variance of each stock's single index residual. As an option, we let each stock's market sensitivity drift around its single index estimate.

Synthetic markets of many stocks let us test a trading method on many plausible paths, rather than on one historical path. Knowing how the synthetic universe is built also tells us its limits: stocks in different sectors move together only through the market draw.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success.

Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___
